# LeetCode 703 — Kth Largest Element in a Stream bằng Python

**Link đề:** https://leetcode.com/problems/kth-largest-element-in-a-stream/
**Độ khó:** Easy · **Chủ đề:** Design, Heap (Priority Queue), Data Stream, Binary Search Tree

```python
class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        ...

    def add(self, val: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài **thiết kế class** (design): một đối tượng giữ trạng thái giữa nhiều lần gọi `add`.
2. Hiểu "phần tử lớn thứ k" khi có **phần tử trùng** (k-th trong danh sách đã sắp xếp, không phải k-th giá trị khác nhau).
3. Đi từ cách **sắp xếp lại mỗi lần** → **chèn vào list đã sắp xếp** (`bisect`) → **min-heap kích thước k** (`heapq`).
4. Hiểu heap là gì, `heappush`, `heappop`, `heapreplace`, vì sao chỉ cần giữ **k phần tử lớn nhất**.
5. Trace trạng thái heap, chứng minh invariant, phân tích Time/Space, biến thể "top-k".

## 1. Đề bài và ví dụ

Thiết kế class `KthLargest`:
- `KthLargest(k, nums)`: khởi tạo với số nguyên `k` và luồng điểm ban đầu `nums`.
- `add(val)`: thêm `val` vào luồng, rồi **trả về phần tử lớn thứ `k`** trong toàn bộ các điểm hiện có.

"Lớn thứ k" nghĩa là: sắp xếp tất cả giảm dần, lấy phần tử ở vị trí thứ `k` (đếm từ 1), **tính cả trùng lặp**.

**Ví dụ 1:** `k = 3`, `nums = [4, 5, 8, 2]`

| Lệnh | Dãy sau khi thêm (giảm dần) | Lớn thứ 3 |
|---|---|:---:|
| `add(3)` | 8 5 4 3 2 | 4 |
| `add(5)` | 8 5 5 4 3 2 | 5 |
| `add(10)` | 10 8 5 5 4 3 2 | 5 |
| `add(9)` | 10 9 8 5 5 4 3 2 | 8 |
| `add(4)` | 10 9 8 5 5 4 4 3 2 | 8 |

**Ví dụ 2:** `k = 4`, `nums = [7, 7, 7, 7, 8, 3]` → `add(2)=7`, `add(10)=7`, `add(9)=7`, `add(9)=8`.

**Constraints:** `0 <= len(nums) <= 10^4`; `1 <= k <= len(nums) + 1`; `-10^4 <= nums[i], val <= 10^4`; tối đa `10^4` lần gọi `add`.

Chú ý `k <= len(nums) + 1`: ngay sau lần `add` đầu tiên luôn có ít nhất `k` phần tử, nên luôn có đáp án.

## 2. Nền tảng Python

### 2.1. `sorted(..., reverse=True)` và chỉ số `k - 1`

Lớn thứ `k` = phần tử ở chỉ số `k - 1` sau khi sắp giảm dần (vì chỉ số bắt đầu từ 0).

In [1]:
scores = [4, 5, 8, 2, 3]
desc = sorted(scores, reverse=True)
print(desc)
k = 3
print("lớn thứ", k, "=", desc[k - 1])

[8, 5, 4, 3, 2]
lớn thứ 3 = 4


### 2.2. Class có trạng thái

Mỗi đối tượng lưu dữ liệu trong thuộc tính `self.xxx`. Gọi phương thức nhiều lần thì dữ liệu được **giữ lại** giữa các lần gọi.

In [2]:
class Counter2:
    def __init__(self, start):
        self.value = start          # trạng thái
    def add(self, x):
        self.value += x
        return self.value

c = Counter2(10)
print(c.add(1), c.add(5), c.add(-2))   # 11 16 14

11 16 14


### 2.3. Mô phỏng chuỗi lệnh kiểu LeetCode

LeetCode cho test dưới dạng hai list: tên lệnh và tham số. Ta viết hàm `run_ops` để chạy giống vậy, dùng lại cho mọi cách cài đặt.

In [3]:
def run_ops(cls, ops, args):
    out = []
    obj = None
    for op, a in zip(ops, args):
        if op == "KthLargest":
            obj = cls(*a)            # *a: "mở" list tham số thành các đối số
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

OPS1 = ["KthLargest", "add", "add", "add", "add", "add"]
ARGS1 = [[3, [4, 5, 8, 2]], [3], [5], [10], [9], [4]]
EXP1 = [None, 4, 5, 5, 8, 8]
OPS2 = ["KthLargest", "add", "add", "add", "add"]
ARGS2 = [[4, [7, 7, 7, 7, 8, 3]], [2], [10], [9], [9]]
EXP2 = [None, 7, 7, 7, 8]
print("đã chuẩn bị 2 ví dụ của đề")

đã chuẩn bị 2 ví dụ của đề


- `cls(*a)` với `a = [3, [4,5,8,2]]` tương đương `cls(3, [4,5,8,2])`.
- `getattr(obj, "add")` lấy phương thức tên `"add"` của `obj`; gọi tiếp `(*a)`.

## 3. Cách 1 — Lưu tất cả, sắp xếp lại mỗi lần `add`

Ý tưởng dễ nhất: giữ list mọi điểm; mỗi `add` thì append, sort giảm dần, lấy `[k-1]`.

In [4]:
class KthLargestSort:
    def __init__(self, k, nums):
        self.k = k
        self.data = list(nums)
    def add(self, val):
        self.data.append(val)
        self.data.sort(reverse=True)
        return self.data[self.k - 1]

print(run_ops(KthLargestSort, OPS1, ARGS1))
print(run_ops(KthLargestSort, OPS2, ARGS2))

[None, 4, 5, 5, 8, 8]
[None, 7, 7, 7, 8]


**Chi phí:** mỗi `add` sắp xếp `m` phần tử (`m` tới `2·10^4`) → `O(m log m)`. `10^4` lần gọi → khoảng `10^4 × 2·10^4 × 15 ≈ 3·10^9` phép trong trường hợp xấu — quá chậm (Timsort của Python nhanh hơn với dữ liệu gần sắp xếp, nhưng ý tưởng vẫn lãng phí).

## 4. Cách 2 — Giữ list luôn sắp xếp, chèn bằng `bisect`

Không cần sắp lại toàn bộ: list đã sắp **tăng dần**, chỉ cần tìm vị trí chèn bằng **binary search** (`O(log m)`) rồi chèn (`O(m)` do phải dịch phần tử).

`bisect.insort(a, x)` chèn `x` vào list tăng dần `a` mà vẫn giữ thứ tự. Lớn thứ `k` trong list tăng dần là `a[-k]`.

In [5]:
import bisect

a = [2, 4, 5, 8]
bisect.insort(a, 3)
print(a)                 # [2, 3, 4, 5, 8]
print(bisect.bisect_left(a, 5), bisect.bisect_right(a, 5))   # vị trí chèn trái / phải của 5
print("a[-3] =", a[-3])  # lớn thứ 3

[2, 3, 4, 5, 8]
3 4
a[-3] = 4


In [6]:
class KthLargestBisect:
    def __init__(self, k, nums):
        self.k = k
        self.data = sorted(nums)
    def add(self, val):
        bisect.insort(self.data, val)
        return self.data[-self.k]

print(run_ops(KthLargestBisect, OPS1, ARGS1))
print(run_ops(KthLargestBisect, OPS2, ARGS2))

[None, 4, 5, 5, 8, 8]
[None, 7, 7, 7, 8]


## 5. Nút thắt và insight

Cả hai cách trên đều **giữ mọi phần tử**. Nhưng hỏi: phần tử nào có thể là đáp án trong tương lai?

- Nếu một điểm `x` đã có **ít nhất k điểm khác ≥ x** thì `x` không bao giờ lọt vào top-k nữa (luồng chỉ thêm, không xóa).
- Vậy chỉ cần giữ **k phần tử lớn nhất**. Đáp án = **phần tử nhỏ nhất** trong nhóm k này.

Ta cần cấu trúc dữ liệu:
- lấy **min** nhanh,
- **xóa min** nhanh (khi nhóm vượt k phần tử),
- **thêm** nhanh.

Đó chính là **min-heap**.

In [7]:
data = [4, 5, 8, 2, 3]
k = 3
top_k = sorted(data, reverse=True)[:k]
print("k phần tử lớn nhất:", top_k, "-> min của chúng:", min(top_k))

k phần tử lớn nhất: [8, 5, 4] -> min của chúng: 4


## 6. Nền tảng: heap và module `heapq`

**Min-heap** là cây nhị phân "gần đầy đủ" lưu trong list, thỏa: mỗi node `<=` các con. Hệ quả: **phần tử nhỏ nhất luôn ở `heap[0]`**.

Với list `h`, con của vị trí `i` nằm ở `2i+1` và `2i+2`. Python cung cấp module `heapq`:

| Hàm | Ý nghĩa | Độ phức tạp |
|---|---|---|
| `heapq.heapify(h)` | biến list thành heap tại chỗ | `O(n)` |
| `heapq.heappush(h, x)` | thêm `x` | `O(log n)` |
| `heapq.heappop(h)` | lấy và xóa min | `O(log n)` |
| `h[0]` | xem min | `O(1)` |
| `heapq.heapreplace(h, x)` | pop min rồi push `x` (một bước) | `O(log n)` |
| `heapq.heappushpop(h, x)` | push `x` rồi pop min (một bước) | `O(log n)` |

In [8]:
import heapq

h = [8, 5, 4, 2, 3]
heapq.heapify(h)
print("sau heapify:", h, "| min =", h[0])
heapq.heappush(h, 1)
print("push 1:", h, "| min =", h[0])
print("pop ->", heapq.heappop(h), "| còn:", h)
print("pop ->", heapq.heappop(h), "| còn:", h)

sau heapify: [2, 3, 4, 5, 8] | min = 2
push 1: [1, 3, 2, 5, 8, 4] | min = 1
pop -> 1 | còn: [2, 3, 4, 5, 8]
pop -> 2 | còn: [3, 5, 4, 8]


Chú ý: list heap **không** được sắp xếp hoàn toàn (`[2, 3, 4, 8, 5]` không tăng dần), chỉ đảm bảo `h[0]` là nhỏ nhất. Đây là lý do heap nhanh hơn sorted list: nó làm ít việc hơn.

Khác nhau giữa `heapreplace` và `heappushpop`:

In [9]:
h = [4, 5, 8]
print(heapq.heappushpop(list(h), 1))   # push 1 trước rồi pop -> trả 1 (chính nó nhỏ nhất)
print(heapq.heapreplace(list(h), 1))   # pop 4 trước rồi push 1 -> trả 4

1
4


## 7. Cách 3 — Min-heap kích thước k (lời giải chính)

Mổ xẻ từng phần:

**`__init__`:**
1. `self.k = k`, `self.heap = list(nums)`, `heapify`.
2. Trong khi heap có hơn `k` phần tử → `heappop` (bỏ các phần tử nhỏ, không bao giờ là đáp án).

**`add(val)`:**
1. Nếu heap chưa đủ `k` phần tử → `heappush(val)`.
2. Ngược lại nếu `val > heap[0]` → `val` lọt top-k, đẩy min hiện tại ra: `heapreplace(heap, val)`.
3. Nếu `val <= heap[0]` → `val` không lọt top-k, bỏ qua.
4. Trả `heap[0]`.

In [10]:
class KthLargestHeap:
    def __init__(self, k, nums):
        self.k = k
        self.heap = list(nums)
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)

    def add(self, val):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, val)
        elif val > self.heap[0]:
            heapq.heapreplace(self.heap, val)
        return self.heap[0]

print(run_ops(KthLargestHeap, OPS1, ARGS1))
print(run_ops(KthLargestHeap, OPS2, ARGS2))

[None, 4, 5, 5, 8, 8]
[None, 7, 7, 7, 8]


Vì sao ở `add` cần nhánh `len(self.heap) < self.k`? Do `k <= len(nums) + 1`, có thể ban đầu `nums` chỉ có `k - 1` phần tử (thậm chí rỗng khi `k = 1`). Lần `add` đầu tiên phải **thêm** chứ không so sánh với `heap[0]` (heap rỗng thì `heap[0]` lỗi `IndexError`).

## 8. TRACE từng bước — Ví dụ 1 (`k = 3`, `nums = [4, 5, 8, 2]`)

| Bước | Thao tác | Quyết định | Heap (nội dung, min ở đầu) | Trả về |
|---|---|---|---|:---:|
| init | heapify [4,5,8,2] | | [2,4,8,5] | |
| init | size 4 > 3 → pop 2 | bỏ 2 | [4,5,8] | |
| 1 | add(3) | 3 ≤ 4 → bỏ qua | [4,5,8] | 4 |
| 2 | add(5) | 5 > 4 → thay 4 | [5,5,8] | 5 |
| 3 | add(10) | 10 > 5 → thay 5 | [5,8,10] | 5 |
| 4 | add(9) | 9 > 5 → thay 5 | [8,9,10] | 8 |
| 5 | add(4) | 4 ≤ 8 → bỏ qua | [8,9,10] | 8 |

Hàm trace in `sorted(heap)` để dễ đọc (thứ tự thật trong list heap có thể khác).

In [11]:
class KthLargestTrace(KthLargestHeap):
    def __init__(self, k, nums):
        self.k = k
        self.heap = list(nums)
        heapq.heapify(self.heap)
        print(f"init: heapify {nums} -> heap={self.heap}")
        while len(self.heap) > k:
            x = heapq.heappop(self.heap)
            print(f"init: size>{k}, pop {x} -> top-k = {sorted(self.heap)}")

    def add(self, val):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, val)
            action = "chưa đủ k -> push"
        elif val > self.heap[0]:
            old = heapq.heapreplace(self.heap, val)
            action = f"{val} > min {old} -> thay {old}"
        else:
            action = f"{val} <= min {self.heap[0]} -> bỏ qua"
        print(f"add({val:>2}): {action:24} | top-k = {sorted(self.heap)} | trả {self.heap[0]}")
        return self.heap[0]

run_ops(KthLargestTrace, OPS1, ARGS1)

init: heapify [4, 5, 8, 2] -> heap=[2, 4, 8, 5]
init: size>3, pop 2 -> top-k = [4, 5, 8]
add( 3): 3 <= min 4 -> bỏ qua     | top-k = [4, 5, 8] | trả 4
add( 5): 5 > min 4 -> thay 4      | top-k = [5, 5, 8] | trả 5
add(10): 10 > min 5 -> thay 5     | top-k = [5, 8, 10] | trả 5
add( 9): 9 > min 5 -> thay 5      | top-k = [8, 9, 10] | trả 8
add( 4): 4 <= min 8 -> bỏ qua     | top-k = [8, 9, 10] | trả 8


[None, 4, 5, 5, 8, 8]

Trace ví dụ 2 — có nhiều phần tử **trùng** (`7, 7, 7, 7`) và `k = 4`:

In [12]:
run_ops(KthLargestTrace, OPS2, ARGS2)

init: heapify [7, 7, 7, 7, 8, 3] -> heap=[3, 7, 7, 7, 8, 7]
init: size>4, pop 3 -> top-k = [7, 7, 7, 7, 8]
init: size>4, pop 7 -> top-k = [7, 7, 7, 8]
add( 2): 2 <= min 7 -> bỏ qua     | top-k = [7, 7, 7, 8] | trả 7
add(10): 10 > min 7 -> thay 7     | top-k = [7, 7, 8, 10] | trả 7
add( 9): 9 > min 7 -> thay 7      | top-k = [7, 8, 9, 10] | trả 7
add( 9): 9 > min 7 -> thay 7      | top-k = [8, 9, 9, 10] | trả 8


[None, 7, 7, 7, 8]

Trace trường hợp biên `k = 1`, `nums = []` (heap rỗng lúc đầu):

In [13]:
run_ops(KthLargestTrace, ["KthLargest", "add", "add", "add"], [[1, []], [-3], [-2], [-4]])

init: heapify [] -> heap=[]
add(-3): chưa đủ k -> push        | top-k = [-3] | trả -3
add(-2): -2 > min -3 -> thay -3   | top-k = [-2] | trả -2
add(-4): -4 <= min -2 -> bỏ qua   | top-k = [-2] | trả -2


[None, -3, -2, -2]

## 9. Code nộp LeetCode

In [14]:
import heapq
from typing import List

class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        self.k = k
        self.heap = nums[:]
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)

    def add(self, val: int) -> int:
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, val)
        elif val > self.heap[0]:
            heapq.heapreplace(self.heap, val)
        return self.heap[0]

print(run_ops(KthLargest, OPS1, ARGS1) == EXP1)
print(run_ops(KthLargest, OPS2, ARGS2) == EXP2)

True
True


`nums[:]` tạo **bản sao** của list (slice toàn bộ) để không làm thay đổi list gốc người gọi truyền vào.

Một cách viết `add` gọn hơn (luôn push rồi pop nếu thừa):

```python
def add(self, val: int) -> int:
    heapq.heappush(self.heap, val)
    if len(self.heap) > self.k:
        heapq.heappop(self.heap)
    return self.heap[0]
```
Đúng, nhưng làm 2 thao tác `O(log k)` mỗi lần; bản chính bỏ qua sớm khi `val <= heap[0]`.

## 10. Vì sao đúng?

**Invariant:** sau `__init__` và sau mỗi `add`, `heap` chứa **đúng `min(k, tổng số điểm)` phần tử lớn nhất** (tính cả trùng) trong toàn bộ luồng đến hiện tại.

- `__init__`: heapify rồi pop phần tử nhỏ nhất cho tới khi còn `k` → còn lại k phần tử lớn nhất. ✔
- `add(val)` khi heap đủ `k`:
  - Nếu `val > heap[0]`: top-k mới = top-k cũ bỏ phần tử nhỏ nhất (`heap[0]`) + `val`. Đúng điều `heapreplace` làm.
  - Nếu `val <= heap[0]`: đã có `k` phần tử `>= val`, nên `val` không thuộc top-k (khi bằng nhau, giữ cái nào cũng cho cùng tập giá trị). Heap giữ nguyên. ✔
- Khi heap chứa đúng k phần tử lớn nhất, phần tử lớn thứ k chính là **nhỏ nhất** trong số đó = `heap[0]`. ✔

Vì đề đảm bảo `k <= len(nums) + 1`, sau mỗi `add` heap luôn có đủ `k` phần tử.

## 11. Độ phức tạp và so sánh

Gọi `n = len(nums)`, `m` = tổng số phần tử hiện có, `q` = số lần `add`.

| Cách | `__init__` | mỗi `add` | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Sort lại mỗi lần | `O(n)` | `O(m log m)` | `O(m)` | Dễ, chậm |
| 2. `bisect.insort` | `O(n log n)` | `O(m)` (dịch list) | `O(m)` | Tìm `O(log m)` nhưng chèn `O(m)` |
| 3. Min-heap size k | `O(n + (n-k) log n)` | `O(log k)` | `O(k)` | **Nên nộp** |

Với `n, q ≤ 10^4`: cách 3 tốn khoảng `10^4 × log₂(10^4) ≈ 1.4·10^5` thao tác cho mọi lần `add`.

In [15]:
import random, time
rng = random.Random(703)
nums_big = [rng.randint(-10**4, 10**4) for _ in range(10**4)]
vals_big = [rng.randint(-10**4, 10**4) for _ in range(10**4)]
for cls in [KthLargestSort, KthLargestBisect, KthLargest]:
    t0 = time.perf_counter()
    obj = cls(500, nums_big)
    last = [obj.add(v) for v in vals_big[:3000]]
    print(f"{cls.__name__:18} 3000 lần add: {time.perf_counter() - t0:.3f}s | kết quả cuối: {last[-1]}")

KthLargestSort     3000 lần add: 0.206s | kết quả cuối: 9264
KthLargestBisect   3000 lần add: 0.006s | kết quả cuối: 9264
KthLargest         3000 lần add: 0.003s | kết quả cuối: 9264


## 12. Bộ test — mẫu, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:34} PASS")

check("mẫu 1", run_ops(KthLargest, OPS1, ARGS1), EXP1)
check("mẫu 2 (trùng lặp)", run_ops(KthLargest, OPS2, ARGS2), EXP2)
check("k=1, nums rỗng", run_ops(KthLargest, ["KthLargest", "add", "add"], [[1, []], [-3], [-2]]), [None, -3, -2])
check("nums có k-1 phần tử", run_ops(KthLargest, ["KthLargest", "add", "add"], [[2, [0]], [-1], [1]]), [None, -1, 0])
check("số âm", run_ops(KthLargest, ["KthLargest", "add", "add"], [[2, [-5, -1]], [-3], [-10]]), [None, -3, -3])
check("toàn giá trị trùng", run_ops(KthLargest, ["KthLargest", "add", "add"], [[3, [5, 5, 5]], [5], [5]]), [None, 5, 5])
check("không làm đổi nums gốc", (lambda L: (KthLargest(1, L), L)[1])([3, 1, 2]), [3, 1, 2])

mẫu 1                              PASS
mẫu 2 (trùng lặp)                  PASS
k=1, nums rỗng                     PASS
nums có k-1 phần tử                PASS
số âm                              PASS
toàn giá trị trùng                 PASS
không làm đổi nums gốc             PASS


In [17]:
for trial in range(300):
    k = rng.randint(1, 8)
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(k - 1, 12))]
    vals = [rng.randint(-20, 20) for _ in range(rng.randint(1, 15))]
    objs = [cls(k, nums) for cls in (KthLargestSort, KthLargestBisect, KthLargestHeap, KthLargest)]
    pool = list(nums)
    for v in vals:
        pool.append(v)
        expected = sorted(pool, reverse=True)[k - 1]
        for o in objs:
            assert o.add(v) == expected
print("300 kịch bản ngẫu nhiên: 4 cách đều khớp oracle sorted()")

300 kịch bản ngẫu nhiên: 4 cách đều khớp oracle sorted()


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng **max-heap** / lấy max thay vì min
Trong top-k ta cần phần tử **nhỏ nhất** (lớn thứ k), nên dùng **min-heap** kích thước k. Nhiều người nghĩ "tìm lớn → max-heap" rồi phải pop k lần mỗi `add` (`O(k log m)`).

### Lỗi 2 — Lớn thứ k theo **giá trị khác nhau** (dùng `set`)

In [18]:
class WrongDistinct:
    def __init__(self, k, nums):
        self.k, self.data = k, set(nums)
    def add(self, val):
        self.data.add(val)
        return sorted(self.data, reverse=True)[self.k - 1]

print("sai :", run_ops(WrongDistinct, OPS2, ARGS2))
print("đúng:", EXP2)

sai : [None, 2, 3, 7, 7]
đúng: [None, 7, 7, 7, 8]


Vì sao sai: `set` gộp `7, 7, 7, 7` thành một, nên sau `add(2)` tập chỉ còn `{2, 3, 7, 8}` và "lớn thứ 4" thành `2` thay vì `7`; các bước sau cũng lệch. Nếu số giá trị khác nhau ít hơn `k` còn bị `IndexError`. Đề đếm **cả phần tử trùng**. Sửa: dùng list/heap, không dùng set.

### Lỗi 3 — Quên trường hợp heap chưa đủ k

In [19]:
class WrongNoSizeCheck:
    def __init__(self, k, nums):
        self.k, self.heap = k, list(nums)
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)
    def add(self, val):
        if val > self.heap[0]:                 # heap có thể rỗng!
            heapq.heapreplace(self.heap, val)
        return self.heap[0]

try:
    run_ops(WrongNoSizeCheck, ["KthLargest", "add"], [[1, []], [5]])
except IndexError as e:
    print("Lỗi:", e)

Lỗi: list index out of range


Vì sao sai: `nums = []`, `k = 1` → `self.heap[0]` không tồn tại. Kể cả khi không lỗi (heap có `k-1` phần tử), so sánh rồi **thay** sẽ làm heap thiếu mãi một phần tử. Sửa: `if len(heap) < k: heappush`.

### Lỗi 4 — Dùng `heappushpop` nhưng không kiểm tra kích thước
`heappushpop` luôn pop → heap không bao giờ tăng kích thước; nếu bắt đầu thiếu thì thiếu mãi.

### Lỗi 5 — Đọc `heap[-1]` tưởng là max
List heap không sắp xếp; `heap[-1]` **không** chắc là lớn nhất. Chỉ `heap[0]` có ý nghĩa (min).

In [20]:
h = [1, 9, 2, 10, 11, 3]
heapq.heapify(h)
print(h, "| h[-1] =", h[-1], "| max thật =", max(h))

[1, 9, 2, 10, 11, 3] | h[-1] = 3 | max thật = 11


## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu "top-k / lớn thứ k / nhỏ thứ k", đặc biệt với dữ liệu đến dần (stream):**
- Lớn thứ k → **min-heap kích thước k** (đáp án ở đỉnh).
- Nhỏ thứ k → **max-heap kích thước k** (Python: đẩy `-x` vào `heapq`).
- Một lần trên mảng cố định có thể dùng `heapq.nlargest(k, nums)` hoặc quickselect.

Bài liên quan: 215 Kth Largest Element in an Array, 347 Top K Frequent Elements, 973 K Closest Points to Origin, 295 Find Median from Data Stream (hai heap).

### 14.1. Biến thể A — `heapq.nlargest` / `nsmallest`

In [21]:
data = [4, 5, 8, 2, 3, 5, 10]
print(heapq.nlargest(3, data))     # [10, 8, 5]
print(heapq.nsmallest(3, data))    # [2, 3, 4]

[10, 8, 5]
[2, 3, 4]


### 14.2. Biến thể B — Kth **Smallest** in a stream (max-heap bằng số âm)

`heapq` chỉ có min-heap. Mẹo: lưu `-x`; khi đó `-heap[0]` là **lớn nhất** trong nhóm.

In [22]:
class KthSmallest:
    def __init__(self, k, nums):
        self.k = k
        self.heap = []
        for x in nums:
            self.add(x)
    def add(self, val):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, -val)
        elif val < -self.heap[0]:
            heapq.heapreplace(self.heap, -val)
        return -self.heap[0]

ks = KthSmallest(2, [5, 1, 9])
print(ks.add(3), ks.add(0), ks.add(7))   # nhỏ thứ 2: 3, 1, 1

3 1 1


## 15. Bài tập tự luyện

**Bài 1 (tính tay).** `k = 2`, `nums = [1]`, lần lượt `add(-1)`, `add(1)`, `add(-2)`, `add(-4)`, `add(3)`. Ghi list kết quả vào `ex1_answers`.

**Bài 2 (đoán output).** Sau `h = [5, 3, 8, 1]; heapq.heapify(h); heapq.heappush(h, 0)`, giá trị `h[0]` là gì? Gán vào `ex2_min`.

**Bài 3 (TODO).** Viết lại class `Ex3KthLargest` bằng min-heap.

**Bài 4 (biến thể).** `ex4_kth_largest_array(nums, k)`: LeetCode 215 dùng heap kích thước k.

**Bài 5 (biến thể).** `ex5_top_k_frequent(nums, k)`: k giá trị xuất hiện nhiều nhất (trả list bất kỳ thứ tự).

**Bài 6 (mini challenge).** Class `Ex6RunningMedian` với `add(x)` trả về **trung vị** hiện tại (dùng hai heap: max-heap nửa dưới, min-heap nửa trên). Với số lượng chẵn trả trung bình hai số giữa (float).

In [23]:
ex1_answers = None   # TODO
ex2_min = None       # TODO

class Ex3KthLargest:
    def __init__(self, k, nums):
        # TODO
        pass
    def add(self, val):
        # TODO
        pass

# assert ex1_answers == [...]
# assert run_ops(Ex3KthLargest, OPS1, ARGS1) == EXP1

In [24]:
def ex4_kth_largest_array(nums, k):
    # TODO
    pass

def ex5_top_k_frequent(nums, k):
    # TODO
    pass

class Ex6RunningMedian:
    def __init__(self):
        # TODO
        pass
    def add(self, x):
        # TODO
        pass

# assert ex4_kth_largest_array([3, 2, 1, 5, 6, 4], 2) == 5
# assert sorted(ex5_top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
# m = Ex6RunningMedian(); assert [m.add(x) for x in [5, 15, 1, 3]] == [5, 10.0, 5, 4.0]

## 16. Gợi ý 3 mức

- **Mức 1:** Lớn thứ k = min của top-k. Trung vị = đỉnh của "nửa dưới" và "nửa trên".
- **Mức 2:** Bài 5: đếm bằng `Counter`, rồi giữ min-heap kích thước k theo **tần suất** (đẩy cặp `(freq, value)`). Bài 6: `low` là max-heap (lưu số âm), `high` là min-heap; luôn đảm bảo `len(low) == len(high)` hoặc `len(low) == len(high) + 1`.
- **Mức 3:** Bài 6 khi thêm `x`: đẩy `-x` vào `low`; chuyển `-heappop(low)` sang `high`; nếu `len(high) > len(low)` thì chuyển `heappop(high)` về `low` (đổi dấu). Trung vị là `-low[0]` (lẻ) hoặc `(-low[0] + high[0]) / 2` (chẵn).

## 17. Đáp án tham khảo

Bài 1: top-2 lần lượt: `[1,-1]`→ -1; `[1,1]`→ 1; `[1,1]`→ 1; `[1,1]`→ 1; `[3,1]`→ 1. `ex1_answers = [-1, 1, 1, 1, 1]`.
Bài 2: `0` (min-heap luôn để nhỏ nhất ở đầu).

In [25]:
from collections import Counter

ans_ex1 = [-1, 1, 1, 1, 1]
ans_ex2 = 0

class AnsEx3KthLargest:
    def __init__(self, k, nums):
        self.k, self.h = k, []
        for x in nums:
            self.add(x)
    def add(self, val):
        if len(self.h) < self.k:
            heapq.heappush(self.h, val)
        elif val > self.h[0]:
            heapq.heapreplace(self.h, val)
        return self.h[0]

def ans_ex4_kth_largest_array(nums, k):
    h = []
    for x in nums:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)
    return h[0]

def ans_ex5_top_k_frequent(nums, k):
    h = []
    for value, freq in Counter(nums).items():
        heapq.heappush(h, (freq, value))
        if len(h) > k:
            heapq.heappop(h)
    return [v for _, v in h]

class AnsEx6RunningMedian:
    def __init__(self):
        self.low, self.high = [], []
    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

o = AnsEx3KthLargest(2, [1])
assert [o.add(v) for v in [-1, 1, -2, -4, 3]] == ans_ex1
h = [5, 3, 8, 1]; heapq.heapify(h); heapq.heappush(h, 0)
assert h[0] == ans_ex2
assert run_ops(AnsEx3KthLargest, OPS1, ARGS1) == EXP1
assert run_ops(AnsEx3KthLargest, OPS2, ARGS2) == EXP2
assert ans_ex4_kth_largest_array([3, 2, 1, 5, 6, 4], 2) == 5
assert sorted(ans_ex5_top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
m = AnsEx6RunningMedian()
assert [m.add(x) for x in [5, 15, 1, 3]] == [5, 10.0, 5, 4.0]
for _ in range(200):
    arr = [rng.randint(-50, 50) for _ in range(rng.randint(1, 30))]
    m = AnsEx6RunningMedian()
    for i, x in enumerate(arr):
        got = m.add(x)
        s = sorted(arr[:i + 1]); L = len(s)
        exp = s[L // 2] if L % 2 else (s[L // 2 - 1] + s[L // 2]) / 2
        assert got == exp
    k = rng.randint(1, len(arr))
    assert ans_ex4_kth_largest_array(arr, k) == sorted(arr, reverse=True)[k - 1]
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh và checklist

```python
import heapq

class KthLargest:
    def __init__(self, k: int, nums: List[int]):
        self.k = k
        self.heap = nums[:]
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)

    def add(self, val: int) -> int:
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, val)
        elif val > self.heap[0]:
            heapq.heapreplace(self.heap, val)
        return self.heap[0]
```

Câu thần chú: **"Lớn thứ k → giữ k thằng lớn nhất trong min-heap, đáp án là đỉnh heap."**

### Checklist trước khi submit
- [ ] Dùng **min**-heap, kích thước tối đa `k`.
- [ ] `__init__` pop bớt cho còn `k` phần tử.
- [ ] `add` xử lý trường hợp heap **chưa đủ k** (kể cả `nums` rỗng).
- [ ] Đếm cả phần tử trùng (không dùng `set`).
- [ ] Trả `self.heap[0]`, không trả `heap[-1]`.
- [ ] Time mỗi `add` `O(log k)`, Space `O(k)`.